# Multi-sample LLM evaluation on StackMathQA

Testing pipeline that evaluates our three solver models on sampled questions from StackMathQA.

## Setup

In [14]:
from datasets import load_dataset
import sympy as sp
import re
import json
import os
import gc
from datetime import datetime
import random
import pandas as pd
from IPython.display import display, Markdown

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

CUDA available: True
GPU name: NVIDIA RTX 5000 Ada Generation


In [2]:
from huggingface_hub import notebook_login

notebook_login()

## Config

In [ ]:
# ---------------------------------------------------------
# CONFIG — edit here
# ---------------------------------------------------------

# Solver models to test. 
# These are all instruct models that fit comfortably on a single
# RTX 5000 Ada (32 GB) in fp16/bf16. 
SOLVER_MODELS = [
    "meta-llama/Meta-Llama-3.1-8B-Instruct",   
    "Qwen/Qwen2.5-7B-Instruct",                # strong general 7B
    "Qwen/Qwen2.5-Math-7B-Instruct",           # math-specialised
]

N_SAMPLES   = 100      # how many questions to evaluate per model
SEED        = 42      # reproducible sample selection
MAX_NEW_TOK = 800    # generation length for the solver
OUTPUT_DIR  = "results"

os.makedirs(OUTPUT_DIR, exist_ok=True)

## Load and filter data

In [ ]:
# Load our dataset from Huggingface
data = load_dataset("math-ai/StackMathQA", "stackmathqa200k")
QandA = data["train"].to_pandas()

# keep only english questions
QandA_eng = QandA[QandA["meta"].apply(lambda x: x.get("language") == "en")].reset_index(drop=True)
print("English Q&A pairs:", len(QandA_eng))

# Fix the sample of questions ONCE so every model sees the same set.
eval_set = QandA_eng.sample(N_SAMPLES, random_state=SEED).reset_index(drop=True)
print("Evaluation set size:", len(eval_set))

English Q&A pairs: 200000
Evaluation set size: 100


## Core helpers

In [ ]:
# ---------------------------------------------------------
# Generation that uses the chat template (no prompt echo)
# ---------------------------------------------------------
def generate_chat(messages, model, tokenizer, max_new_tokens=512):
    """messages: list of {"role","content"} dicts. Returns ONLY new tokens."""
    
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)
    print(inputs["input_ids"].shape)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,                 # deterministic
            pad_token_id=tokenizer.eos_token_id,
        )
    # We only output the newly generated tokens
    prompt_len =inputs["input_ids"].shape[1]
    gen_tokens = outputs[0][prompt_len:]
    
    return tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()

In [ ]:
# ---------------------------------------------------------
# Extract final answer — anchored to end of line, ignores the
# template placeholder, takes the LAST occurrence if several.
# This was an attempt to extract models final answer
# ---------------------------------------------------------
def extract_final_answer(text):
    matches = re.findall(r"Final answer:\s*(.+)", text, flags=re.IGNORECASE)
    if matches:
        ans = matches[-1].strip()
        # guard against the model parroting the literal placeholder
        if ans and ans.lower() not in ("<answer>", "answer"):
            return ans
    # fallback: a boxed answer, common in math models
    boxed = re.findall(r"\\boxed\{([^}]*)\}", text)
    if boxed:
        return boxed[-1].strip()
    return ""   # nothing parseable


# ---------------------------------------------------------
# Symbolic equivalence — only meaningful for short symbolic
# answers. StackMathQA "A" fields are often long prose, so this
# will usually be inconclusive; 
# First atempt of correctness evaluation, but it turned out poorly
# ---------------------------------------------------------
def equivalent(a, b):
    if not a or not b:
        return False
    try:
        ea = sp.sympify(a)
        eb = sp.sympify(b)
        return sp.simplify(ea - eb) == 0
    except Exception:
        return a.replace(" ", "") == b.replace(" ", "")

## Solving prompt

In [10]:
SOLVE_SYSTEM = "You are a helpful assistant for solving math problems."

def solve_one(sample, model, tokenizer, max_new_tokens=512):
    question = sample["Q"]
    correct_answer = sample["A"]

    user_prompt = f"""Solve the following problem using a clear, structured approach.

    RULES YOU MUST FOLLOW:
    1. Show your reasoning step by step.
    2. Keep your explanation clear and concise (max 8-10 sentences).
    3. If you dont have enough information to solve, say so instead of guessing and trying random calculations.
    4. If the problem is a proof, provide a short, correct proof. Do not write a long essay.
    5. If the problem is computationL, compute the answer directly without writing a long essay.
    6. If you cannot determine the answer, write:
    "Final answer: I cannot determine the answer with the given information."
    7. ALWAYS end with a line:
    "Final answer: <your final answer here>"
    
    Problem:
    {question}"""

    messages = [
        {"role": "system", "content": SOLVE_SYSTEM},
        {"role": "user", "content": user_prompt},
    ]
    model_output = generate_chat(messages, model, tokenizer, max_new_tokens)
    final_answer = extract_final_answer(model_output)
    is_correct = equivalent(final_answer, correct_answer)

    return {
        "question": question,
        "correct_answer": correct_answer,
        "explanation": model_output,
        "final_answer": final_answer,
        "is_correct": bool(is_correct),
    }

## EVALUATE MODELS

### Run loop — all models, all samples, save outputs

In [29]:
def load_model(name):
    print(f"  loading {name} ...")
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    mdl = AutoModelForCausalLM.from_pretrained(
        name, dtype=torch.float16, device_map="cuda"
    )
    mdl.eval()
    return mdl, tok


def free_model(mdl):
    del mdl
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# ===== SAVE =====
# Full records (including evaluated question + model whole explanation) -> JSONL
def save_results(solve_records, model_name):
    

    run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
    jsonl_path = os.path.join(OUTPUT_DIR, f"eval_fullnew_{run_id}_{model_name}.jsonl")
    csv_path   = os.path.join(OUTPUT_DIR, f"eval_summarynew_{run_id}_{model_name}.csv")

    os.makedirs(os.path.dirname(jsonl_path), exist_ok=True)
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    
    with open(jsonl_path, "w", encoding="utf-8") as f:
        for rec in solve_records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    # Compact summary -> CSV (truncate the huge text fields for readability)
    df = pd.DataFrame(solve_records)
    summary_cols = (
        ["question", "correct_answer", "explanation", "final_answer", "is_correct"]
    )
    df_summary = df[summary_cols].copy()
    df_summary.to_csv(csv_path, index=False)

    print("Saved:")
    print(" ", jsonl_path)
    print(" ", csv_path)

In [ ]:
# ===== SOLVE all questions for all SOLVER MODELS =====
# This took around 40 min for each model, ran in AI Lab.

for model_name in SOLVER_MODELS:
    solve_records = [] 
    print(f"\n=== SOLVING with {model_name} ===")
    mdl, tok = load_model(model_name)
    for i in range(len(eval_set)):
        sample = eval_set.iloc[i]
        rec = solve_one(sample, mdl, tok, MAX_NEW_TOK)
        rec["model"] = model_name
        rec["sample_idx"] = int(i)
        solve_records.append(rec)
        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(eval_set)} solved")
    free_model(mdl)
    save_results(solve_records, model_name)

print(f"\nSolved {len(solve_records)} (model x sample) pairs total.")


=== SOLVING with Qwen/Qwen2.5-Math-7B-Instruct ===
  loading Qwen/Qwen2.5-Math-7B-Instruct ...


config.json:   0%|          | 0.00/658 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/161 [00:00<?, ?B/s]

torch.Size([1, 448])
torch.Size([1, 397])
torch.Size([1, 369])
torch.Size([1, 388])
torch.Size([1, 434])
torch.Size([1, 1700])
torch.Size([1, 325])
torch.Size([1, 990])
torch.Size([1, 326])
torch.Size([1, 340])
  10/100 solved
torch.Size([1, 505])
torch.Size([1, 233])
torch.Size([1, 270])
torch.Size([1, 1279])
torch.Size([1, 290])
torch.Size([1, 541])
torch.Size([1, 519])
torch.Size([1, 392])
torch.Size([1, 728])
torch.Size([1, 381])
  20/100 solved
torch.Size([1, 688])
torch.Size([1, 460])
torch.Size([1, 310])
torch.Size([1, 470])
torch.Size([1, 384])
torch.Size([1, 841])
torch.Size([1, 330])
torch.Size([1, 856])
torch.Size([1, 801])
torch.Size([1, 1058])
  30/100 solved
torch.Size([1, 253])
torch.Size([1, 290])
torch.Size([1, 483])
torch.Size([1, 453])
torch.Size([1, 680])
torch.Size([1, 344])
torch.Size([1, 364])
torch.Size([1, 379])
torch.Size([1, 320])
torch.Size([1, 607])
  40/100 solved
torch.Size([1, 554])
torch.Size([1, 578])
torch.Size([1, 668])
torch.Size([1, 402])
torch.Siz

KeyboardInterrupt: 

## DISPLAYING MODEL ANSWERS

Look at some model answers in a readable nice format

In [14]:
import pandas as pd
from IPython.display import display, Markdown


In [3]:
jsonl_path = "results/qwen_math_results_full.jsonl"   
df = pd.read_json(jsonl_path, lines=True)

In [4]:
def latex_block(text):
    """
    Convert inline LaTeX like $x^2$ and display blocks like $$...$$
    into Markdown-friendly MathJax.
    """
    # Ensure $$...$$ blocks render properly
    # Ensure $...$ inline math renders properly
    return text.replace("\n", "\n\n")  # helps Markdown formatting

In [7]:
df.columns

Index(['question', 'correct_answer', 'explanation', 'final_answer',
       'is_correct', 'model', 'sample_idx'],
      dtype='object')

In [8]:
df.head()

,question,correct_answer,explanation,final_answer,is_correct,model,sample_idx
0,"For a certain base $b$, the product $(12_b)(15...",Deciding that $b=9$ can be accomplished withou...,To solve for \( s = 12_b + 15_b + 16_b \) in b...,None,False,Qwen/Qwen2.5-Math-7B-Instruct,0
1,The number of integral solution of $\alpha+\be...,One approach would be to first express it as a...,To solve the problem of finding the number of ...,None,False,Qwen/Qwen2.5-Math-7B-Instruct,1
2,Show that $f(x)=x^{5/3}-kx^{4/3}+k^2x$ is incr...,Change variables to $y = x^{1/3}$ to get the q...,To determine whether the function \( f(x) = x^...,None,False,Qwen/Qwen2.5-Math-7B-Instruct,2
3,Inequality $(xy+z)^2+(yz+x)^2+(zx+y)^2\geq\sqr...,"Let $x+y+z=3u$, $xy+xz+yz=3v^2$ and $xyz=w^3$....",To prove the inequality \((xy+z)^2 + (yz+x)^2 ...,None,False,Qwen/Qwen2.5-Math-7B-Instruct,3
4,How many elements of order $2$ does Sym $6$ ha...,Involutions (elements of order 2) in $S_n$ are...,To determine the number of elements of order 2...,75,False,Qwen/Qwen2.5-Math-7B-Instruct,4


In [15]:
def show_sample(idx):
    row = df.iloc[idx]

    question = latex_block(row["question"])
    correct_answer = latex_block(row["correct_answer"])
    explanation = latex_block(row["explanation"])
    final_answer = latex_block(row["final_answer"]) if pd.notna(row["final_answer"]) else "_No answer provided_"

    md = f"""
### 🔢 Sample {idx}

---

### ❓ **Question**
{question}

---

### ✅ **Correct Answer**
{correct_answer}

---

### 🤖 **Model Explanation**
{explanation}

---

### 🎯 **Model Final Answer**
{final_answer}

---

### 🧪 **Correct?**  
**{row['is_correct']}**

---
"""

    display(Markdown(md))

In [16]:
show_sample(0)


### 🔢 Sample 0

---

### ❓ **Question**
For a certain base $b$, the product $(12_b)(15_b)(16_b)$ is equal to $3146_b$. Let $s = 12_b + 15_b + 16_b$. What is $s$ in base $b$? I have worked out the above problem in the following way:

$(12_b)(15_b)(16_b)=(b+2)(b+5)(b+6)=3146_b=3b^3+b^2+4b+6$

$=>b^3+13b^2+52b+60=3b^3+b^2+4b+6$

$=>-2b^3+12b^2+48b+54=0$

$=>-b^3+6b^2+24b+27=0$

$=>6b^2+24b+27=b^3$

=>$2b^2+8b+9=b^3$

We need to find $s=3b+13$. I am stuck there. Any hint on how to proceed would be appreciated.



---

### ✅ **Correct Answer**
Deciding that $b=9$ can be accomplished without resorting to cubic equations, and the answer to the posed question, "What is $s$ in base $b$?" is $44$ ($40$ given in previous answers is the base $10$ representation of $s$).

First, since $6$ is a digit in base $b$, $b>6$.

Next, since $12\times 15\times 16=2880_{10}$, the fact that the representation in base $b$ is $3146$ means that $b<10$. If that isn't apparent, I will explain it after resolving the main question.

Finally, looking only at the units digits, $2\times 5\times 6=60_{10}$, but the units digit in the base $b$ representation is $6$. That means $54_{10}$ units have to be carried out of the units place to a higher place (the $b^2$ place, or possibly even the $b^3$ place). That further means that $54$ is a multiple of $b$

The only conclusion consistent with these facts is $b=9$, and checking the math, it works. Reverting to base $10$ to multiply conveniently, we get $11\times 14\times 15=2310$, which when converted to base $9$ is represented as $3146$.

As an aside, I note that $2310=2\cdot 3\cdot 5\cdot 7\cdot 11$ which is the product of the first $5$ primes, or the primorial $p_5\#$. Thus, $3146_9=p_5\#$.

Now to the understanding that a "larger" appearing representation of a quantity results from being presented in a smaller base. The best way to appreciate this is to look at a fixed quantity, say the quantity we mean when we say "one hundred," in many different bases. In what follows, I will use $t$ to indicate a digit with the value of ten, and $f$ to indicate a digit with the value of fifteen. $$\text{One hundred}= 50_{20}=55_{19}=5t_{18}=5f_{17}=64_{16}=6t_{15}=72_{14}=79_{13}=84_{12}=91_{11}=100_{10}=121_9=144_8=202_7=244_6=400_5=1210_4=10201_3=1100100_2$$

The smaller the base, the "larger" the representation of a fixed quantity appears.



---

### 🤖 **Model Explanation**
To solve for \( s = 12_b + 15_b + 16_b \) in base \( b \), we start by converting each base \( b \) number to base 10. The base \( b \) number \( 12_b \) is equal to \( b + 2 \), the base \( b \) number \( 15_b \) is equal to \( b + 5 \), and the base \( b \) number \( 16_b \) is equal to \( b + 6 \). Therefore, we have:



\[

(12_b)(15_b)(16_b) = (b+2)(b+5)(b+6)

\]



We are given that this product is equal to \( 3146_b \). Converting \( 3146_b \) to base 10, we get:



\[

3146_b = 3b^3 + 1b^2 + 4b + 6

\]



So, we can set up the equation:



\[

(b+2)(b+5)(b+6) = 3b^3 + b^2 + 4b + 6

\]



Next, we expand the left-hand side of the equation:



\[

(b+2)(b+5) = b^2 + 7b + 10

\]

\[

(b^2 + 7b + 10)(b+6) = b^3 + 6b^2 + 7b^2 + 42b + 10b + 60 = b^3 + 13b^2 + 52b + 60

\]



Now, we set the expanded form equal to the right-hand side of the original equation:



\[

b^3 + 13b^2 + 52b + 60 = 3b^3 + b^2 + 4b + 6

\]



To isolate \( b \), we move all terms to one side of the equation:



\[

b^3 + 13b^2 + 52b + 60 - 3b^3 - b^2 - 4b - 6 = 0

\]

\[

-2b^3 + 12b^2 + 48b + 54 = 0

\]

\[

b^3 - 6b^2 - 24b - 27 = 0

\]



We need to find the roots of this polynomial. By the Rational Root Theorem, possible rational roots are the factors of the constant term \(-27\) divided by the factors of the leading coefficient \(1\). The possible rational roots are \( \pm 1, \pm 3, \pm 9, \pm 27 \). We test these possible roots by substitution. Let's try \( b = 9 \):



\[

9^3 - 6 \cdot 9^2 - 24 \cdot 9 - 27 = 729 - 486 - 216 - 27 = 0

\]



Since \( b = 9 \) is a root, we can factor \( b^3 - 6b^2 - 24b - 27 \) as \( (b-9)(b^2 + 3b + 3) \). The quadratic \( b^2 + 3b + 3 \) has no real roots (the discriminant \( 3^2 - 4 \cdot 1 \cdot 3 = 9 - 12 = -3 \) is negative), so the

---

### 🎯 **Model Final Answer**
_No answer provided_

---

### 🧪 **Correct?**  
**False**

---
